# EDA

Initial exploration notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import yaml
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# Charger les paramètres centralisés
with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

print("✅ Imports OK")
print(f"📁 Dataset cible : {params['data']['raw_path']}")

In [ ]:
df = pd.read_csv(f"../{params['data']['raw_path']}")

df.shape
df.head()

In [ ]:
import pandas as pd

# 1. Calcul des métriques
types = df.dtypes
nulls = df.isnull().sum()
percents = (df.isnull().sum() / len(df) * 100).round(2)
uniques = df.nunique()
exemples = df.iloc[0]

# 2. Assemblage propre avec concat
audit = pd.concat([types, nulls, percents, uniques, exemples], axis=1)

# 3. Renommage des colonnes
audit.columns = ['Type', 'Nulls', 'Null_%', 'Uniques', 'Exemple']

# 4. Affichage stylisé
print("=" * 60)
print("INVENTAIRE STRUCTUREL")
print("=" * 60)
print(audit)

In [ ]:
# Adapte 'Catégorie' au nom exact de ta colonne cible
TARGET_COL = 'Catégorie'   # ← vérifie ce nom dans ton dataset

print("=" * 60)
print(f"DISTRIBUTION DE LA CIBLE : {TARGET_COL}")
print("=" * 60)

dist = df[TARGET_COL].value_counts()
dist_pct = df[TARGET_COL].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    'Count'   : dist,
    'Pct_%'   : dist_pct.round(2),
    'Ratio vs majoritaire' : (dist.max() / dist).round(2)
})
print(summary.to_string())

# Ratio de déséquilibre global
ratio = dist.max() / dist.min()
print(f"\n⚠️  Ratio déséquilibre max/min : {ratio:.1f}x")
if ratio > 5:
    print("   → Déséquilibre SÉVÈRE — SMOTE nécessaire")
elif ratio > 2:
    print("   → Déséquilibre MODÉRÉ — class_weight='balanced' suffira peut-être")
else:
    print("   → Dataset relativement équilibré")

In [ ]:
import os
import matplotlib.pyplot as plt

# 1. Gestion automatique du dossier de sauvegarde
output_dir = '../notebooks/figures/'
os.makedirs(output_dir, exist_ok=True) # Crée le dossier s'il n'existe pas
output_path = os.path.join(output_dir, '01_distribution_cible.png')

# 2. Configuration du style et des subplots
plt.style.use('seaborn-v0_8-muted') # Style plus moderne
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- PARTIE GAUGHE : Barplot ---
dist.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white', alpha=0.9)
axes[0].set_title('Distribution des Catégories (Effectifs)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Nombre de tickets', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# Ajout des étiquettes au-dessus des barres
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width()/2., p.get_height()), 
                     ha='center', va='center', 
                     xytext=(0, 9), 
                     textcoords='offset points',
                     fontsize=10, fontweight='bold')

# --- PARTIE DROITE : Pie Chart ---
# Couleurs harmonisées
colors = plt.cm.tab10(range(len(dist))) 
dist.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=140, 
          colors=colors, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Répartition en %', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('') # Supprime le label par défaut 'None'

# 3. Finalisation et Sauvegarde
plt.tight_layout()
try:
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"✅ Figure sauvegardée avec succès ici : {output_path}")
except Exception as e:
    print(f"❌ Erreur lors de la sauvegarde : {e}")

plt.show()

In [ ]:
# Adapte ces noms à tes colonnes réelles
TITLE_COL = 'Titre'
DESC_COL  = 'Description'

# Fusionner pour analyse
df['text_combined'] = (
    df[TITLE_COL].fillna('') + ' ' + df[DESC_COL].fillna('')
).str.strip()

# Longueurs
df['len_title']    = df[TITLE_COL].fillna('').apply(len)
df['len_desc']     = df[DESC_COL].fillna('').apply(len)
df['len_combined'] = df['text_combined'].apply(len)
df['word_count']   = df['text_combined'].apply(lambda x: len(x.split()))

print("=" * 60)
print("STATISTIQUES DE LONGUEUR DES TEXTES")
print("=" * 60)
print(df[['len_title', 'len_desc', 'len_combined', 'word_count']].describe().round(2))

# Tickets quasi-vides
empty_thresh = 5  # moins de 5 mots = quasi-vide
quasi_vides  = df[df['word_count'] < empty_thresh]
print(f"\n⚠️  Tickets quasi-vides (< {empty_thresh} mots) : {len(quasi_vides)}")
if len(quasi_vides) > 0:
    print(quasi_vides[[TITLE_COL, DESC_COL, 'word_count']].head(10).to_string())

In [ ]:
# Visualisation des longueurs
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label in zip(axes,
                           ['len_title', 'len_desc', 'word_count'],
                           ['Longueur Titre', 'Longueur Description', 'Nombre de mots']):
    df[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(label, fontsize=12)
    ax.axvline(df[col].median(), color='red', linestyle='--', label=f'Médiane: {df[col].median():.0f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/02_longueurs_textes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 60)
print("ANALYSE DES DUPLICATAS")
print("=" * 60)

# Duplicatas exacts sur le texte combiné
n_dupl = df.duplicated(subset=['text_combined']).sum()
print(f"Duplicatas exacts (texte combiné) : {n_dupl}")

# Duplicatas sur le titre uniquement
n_dupl_title = df.duplicated(subset=[TITLE_COL]).sum()
print(f"Titres identiques                : {n_dupl_title}")

# Top titres répétés
print(f"\nTop 10 titres les plus répétés :")
print(df[TITLE_COL].value_counts().head(10).to_string())

In [ ]:
print("=" * 60)
print("COHÉRENCE DES LABELS PAR TITRE RÉPÉTÉ")
print("=" * 60)

# Pour chaque titre qui apparaît plusieurs fois,
# vérifier si la catégorie est toujours la même
title_cats = (
    df.groupby(TITLE_COL)[TARGET_COL]
    .nunique()
    .reset_index()
    .rename(columns={TARGET_COL: 'n_categories_differentes'})
)

incoherents = title_cats[title_cats['n_categories_differentes'] > 1]
print(f"Titres identiques avec catégories différentes : {len(incoherents)}")

if len(incoherents) > 0:
    print("\n⚠️  Exemples d'incohérences :")
    exemples = df[df[TITLE_COL].isin(incoherents[TITLE_COL])]\
               [[TITLE_COL, TARGET_COL]]\
               .sort_values(TITLE_COL)
    print(exemples.head(20).to_string())

In [ ]:
# Adapte ces noms à tes colonnes réelles
STATUS_COL     = 'STATUT'          # ← vérifie
RESOLUTION_COL = 'Résolution/Motif'      # ← vérifie (ou 'Motif', 'Solution'...)

print("=" * 60)
print("AUDIT BASE DE CONNAISSANCE RAG")
print("=" * 60)

# Valeurs uniques du statut
print(f"Valeurs uniques de '{STATUS_COL}' :")
print(df[STATUS_COL].value_counts().to_string())

# Filtrer les tickets résolus avec résolution non-vide
mask_resolu  = df[STATUS_COL].str.lower().str.contains('r.solu|closed|fermé', na=False)
mask_solution = df[RESOLUTION_COL].notna() & (df[RESOLUTION_COL].str.strip() != '')

rag_base = df[mask_resolu & mask_solution]

print(f"\n✅ Tickets résolus avec résolution renseignée : {len(rag_base)}")
print(f"   ({len(rag_base)/len(df)*100:.1f}% du dataset total)")

if len(rag_base) < 200:
    print("⚠️  ALERTE : Base RAG trop petite — les recommandations seront limitées")
elif len(rag_base) < 500:
    print("⚠️  Base RAG acceptable mais perfectible")
else:
    print("✅  Base RAG de bonne taille")

# Distribution par catégorie dans la base RAG
print(f"\nDistribution RAG par catégorie :")
print(rag_base[TARGET_COL].value_counts().to_string())

In [ ]:
# Créer un rapport texte de l'audit
rapport = f"""
========================================
RAPPORT D'AUDIT EDA — HELPDESK AI
========================================
Date           : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
Dataset        : {params['data']['raw_path']}
Nb tickets     : {len(df)}
Nb colonnes    : {df.shape[1]}

--- CIBLE ---
Colonne cible  : {TARGET_COL}
Nb classes     : {df[TARGET_COL].nunique()}
Ratio déséquil : {ratio:.1f}x

--- TEXTE ---
Mots médiane   : {df['word_count'].median():.0f}
Tickets vides  : {len(quasi_vides)}
Duplicatas     : {n_dupl}
Labels incoh.  : {len(incoherents)}

--- RAG ---
Base RAG size  : {len(rag_base)} tickets
"""

with open('../notebooks/rapport_audit.txt', 'w', encoding='utf-8') as f:
    f.write(rapport)

print(rapport)
print("✅ Rapport sauvegardé dans notebooks/rapport_audit.txt")

phase2

In [ ]:
import pandas as pd
import numpy as np
import re
import yaml
import os
import unicodedata
import matplotlib.pyplot as plt
from collections import Counter

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords

with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

print("✅ Imports OK")

In [ ]:
df = pd.read_csv(f"../{params['data']['raw_path']}")

# Les 4 colonnes texte disponibles
TEXT_COLS = ['Titre', 'Description', 'Résolution/Motif', 'knowledge Base']

print("=" * 60)
print("APERÇU DES COLONNES TEXTE")
print("=" * 60)

for col in TEXT_COLS:
    print(f"\n--- {col} ---")
    print(f"  Nulls    : {df[col].isnull().sum()}")
    print(f"  Uniques  : {df[col].nunique()}")
    print(f"  Exemple  : {str(df[col].iloc[5])[:120]}")

In [ ]:
# Stop words : français + anglais + domaine IT personnalisé
STOP_FR = set(stopwords.words('french'))
STOP_EN = set(stopwords.words('english'))

# Mots vides spécifiques à ton domaine
# (mots fréquents mais sans valeur discriminante dans les tickets IT)
STOP_IT = {
    # Politesse standard
    'bonjour', 'support', 'merci', 'cordialement',
    'bonne', 'journee', 'svp', 'lors', 'suite',
    'hello', 'hi', 'dear', 'team', 'please', 'regards',
    
    # Formules de clôture des tickets (la source du bruit)
    'avance',          # "merci d'avance"
    'equipe',          # "notre équipe va..."
    'verifier',        # "...va vérifier"
    'faire', 'retour', # "...faire un retour"
    'prendre', 'charge', 'cette',  # "prendre en charge"
    'peux', 'aider', 'resoudre',   # "pouvez-vous aider à résoudre"
    'bien', 'vouloir', 'etudier',
    'probleme',        # trop générique, présent partout
    'question', 'demande',
    
    # Termes IT trop génériques
    'ticket', 'utilisateur', 'utilisateurs',
    'systeme', 'service',  # trop génériques
    
    # Entreprise
    'otbs', 'one', 'tech', 'business', 'solutions',
    'pouvez', 'plait', 'point',
    'sil', 'vous',
}

STOP_WORDS = STOP_FR | STOP_EN | STOP_IT


def clean_text(text: str) -> str:
    """
    Pipeline de nettoyage NLP pour tickets IT.
    
    Étapes :
    1. Conversion en minuscules
    2. Suppression des accents
    3. Suppression des URLs, emails, IPs
    4. Suppression des codes techniques (CID-xxxx, etc.)
    5. Suppression des caractères spéciaux
    6. Suppression des nombres isolés
    7. Suppression des stop words
    8. Suppression des tokens trop courts (< 2 chars)
    9. Normalisation des espaces
    """
    if not isinstance(text, str) or text.strip() == '':
        return ''
    
    # 1. Minuscules
    text = text.lower()
    
    # 2. Suppression des accents
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    
    # 3. Suppression URLs, emails, IPs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', ' ', text)
    
    # 4. Codes techniques (CID-xxxx, SRV-xx, etc.)
    text = re.sub(r'\b[a-z]+-\d+\b', ' ', text)
    text = re.sub(r'\bcid\b', ' ', text)
    
    # 5. Caractères spéciaux → espace
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # 6. Nombres isolés (pas de valeur discriminante)
    text = re.sub(r'\b\d+\b', ' ', text)
    
    # 7. Tokenisation + suppression stop words + tokens courts
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) >= 3]
    
    # 8. Recomposer
    return ' '.join(tokens)


# Test rapide
exemples = [
    "Erreur lors de l'activation d'Office 365",
    "Problème de connexion VPN depuis le bureau",
    "Bonjour support, je n'arrive plus à accéder à Sage X3",
    "RESET MOT DE PASSE",
    "BioTime inaccessible depuis ce matin"
]

print("TESTS DE NETTOYAGE")
print("=" * 60)
for e in exemples:
    print(f"  AVANT : {e}")
    print(f"  APRÈS : {clean_text(e)}")
    print()

In [ ]:
# Vu que les titres sont quasi-identiques (1742/1800),
# on teste 3 stratégies et on compare visuellement

def build_features(row, strategy='desc_only'):
    title = str(row['Titre'])
    desc  = str(row['Description'])
    
    if strategy == 'desc_only':
        # Stratégie 1 : Description seule (recommandée vu l'analyse)
        return clean_text(desc)
    
    elif strategy == 'title_desc':
        # Stratégie 2 : Titre + Description (poids égal)
        return clean_text(title + ' ' + desc)
    
    elif strategy == 'title_weighted':
        # Stratégie 3 : Titre répété 2x pour le pondérer
        return clean_text(title + ' ' + title + ' ' + desc)


# Appliquer les 3 stratégies
for strat in ['desc_only', 'title_desc', 'title_weighted']:
    df[f'text_{strat}'] = df.apply(
        lambda row: build_features(row, strat), axis=1
    )

# Comparer les longueurs
print("COMPARAISON DES STRATÉGIES DE FUSION")
print("=" * 60)
for strat in ['desc_only', 'title_desc', 'title_weighted']:
    col    = f'text_{strat}'
    vides  = (df[col] == '').sum()
    moy    = df[col].apply(lambda x: len(x.split())).mean()
    print(f"  {strat:20s} | mots moyens : {moy:.1f} | vides : {vides}")

In [ ]:
# Décision finale : on garde desc_only comme feature principale
# et on ajoute une colonne text_rag pour le moteur RAG
df['text_clean']   = df['text_desc_only']    # ← feature modèle ML
df['text_rag']     = df.apply(               # ← feature moteur RAG
    lambda row: clean_text(
        str(row['Description']) + ' ' +
        str(row['Résolution/Motif']) + ' ' +
        str(row['knowledge Base'])
    ), axis=1
)

print(f"✅ text_clean   : {df['text_clean'].apply(lambda x: len(x.split())).mean():.1f} mots en moyenne")
print(f"✅ text_rag     : {df['text_rag'].apply(lambda x: len(x.split())).mean():.1f} mots en moyenne")

# Vérification : textes vides
print(f"\nTextes vides text_clean : {(df['text_clean'] == '').sum()}")
print(f"Textes vides text_rag   : {(df['text_rag'] == '').sum()}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("TOP 10 MOTS DISCRIMINANTS PAR CATÉGORIE")
print("=" * 60)

for cat in df['Catégorie'].unique():
    corpus_cat = df[df['Catégorie'] == cat]['text_clean'].tolist()
    
    if len(corpus_cat) < 5:
        continue
    
    tfidf = TfidfVectorizer(max_features=10, ngram_range=(1,2))
    try:
        tfidf.fit(corpus_cat)
        mots = tfidf.get_feature_names_out()
        print(f"\n  [{cat}]")
        print(f"    {', '.join(mots)}")
    except:
        print(f"  [{cat}] → pas assez de données")

In [ ]:
# Colonnes à conserver pour la suite
cols_to_save = [
    'ID', 'STATUT', 'Catégorie',
    'Titre', 'Description',
    'Résolution/Motif', 'knowledge Base',
    'text_clean', 'text_rag'
]

df_clean = df[cols_to_save].copy()

# Sauvegarder
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv(
    f"../{params['data']['processed_path']}",
    index=False,
    encoding='utf-8'
)

print(f"✅ Dataset nettoyé sauvegardé : {params['data']['processed_path']}")
print(f"   Shape : {df_clean.shape}")
print(f"\nAperçu :")
print(df_clean[['Catégorie', 'text_clean']].head(5).to_string())

In [ ]:
print("TOP 10 MOTS DISCRIMINANTS PAR CATÉGORIE")
print("=" * 60)

for cat in df['Catégorie'].unique():
    corpus_cat = df[df['Catégorie'] == cat]['text_clean'].tolist()
    tfidf = TfidfVectorizer(max_features=10, ngram_range=(1,2))
    try:
        tfidf.fit(corpus_cat)
        mots = tfidf.get_feature_names_out()
        print(f"\n  [{cat}]")
        print(f"    {', '.join(mots)}")
    except:
        print(f"  [{cat}] → pas assez de données")

phase3

In [ ]:
import json
from pathlib import Path

candidates = [
    Path("models/model_metadata.json"),
    Path("../models/model_metadata.json"),
]
model_path = next((p for p in candidates if p.exists()), candidates[-1])

if not model_path.exists():
    raise FileNotFoundError(
        f"Fichier introuvable: {model_path}. "
        "Execute la cellule de sauvegarde du modele avant celle-ci."
)

with open(model_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print(json.dumps(meta, ensure_ascii=False, indent=2))

## Benchmark externe — Jeu de données public similaire

Pour situer notre jeu de données et disposer d'un point de comparaison, on charge un
**jeu de données public de tickets de support** (Hugging Face — `Tobi-Bueck/customer-support-tickets`),
filtré sur l'anglais (**11 923 tickets**).

**Correspondance structurelle avec notre schéma :**

| Notre dataset | Benchmark public | Rôle |
|---|---|---|
| `Titre` | `subject` | titre court |
| `Description` | `body` | texte libre |
| `Catégorie` (8 classes) | `queue` (10 classes) | **cible de classification** |
| `Classification` (Demande / Incident…) | `type` (Incident / Request / Problem / Change) | nature du ticket |
| `Priorité` | `priority` | priorité |
| `Résolution/Motif` | `answer` | réponse / résolution |

- **Source** : https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets
- **Fichier local** : `data/external/similar_helpdesk_dataset.csv`
- **Figures produites** : `bench_01_distribution_classes.png`, `bench_02_comparaison.png`

In [ ]:
# ── Benchmark : chargement & aperçu (étape 4) ────────────────────────────────
BENCH_PATH = '../data/external/similar_helpdesk_dataset.csv'
bench = pd.read_csv(BENCH_PATH)

print("=" * 60)
print("BENCHMARK PUBLIC — APERÇU")
print("=" * 60)
print(f"Lignes   : {bench.shape[0]}")
print(f"Colonnes : {bench.shape[1]}")
print(f"Noms     : {list(bench.columns)}")

print("\n--- 5 premières lignes (subject / queue / type / priority) ---")
print(bench[['subject', 'queue', 'type', 'priority']].head(5).to_string())

# Résumé des colonnes
resume = pd.DataFrame({
    'Type'    : bench.dtypes.astype(str),
    'Nulls'   : bench.isnull().sum(),
    'Null_%'  : (bench.isnull().mean() * 100).round(1),
    'Uniques' : bench.nunique(),
})
print("\n--- Résumé des colonnes ---")
print(resume.to_string())

In [ ]:
# ── Benchmark : distribution des classes (étape 4) ───────────────────────────
TARGET_BENCH = 'queue'   # colonne catégorie du benchmark

dist_b     = bench[TARGET_BENCH].value_counts()
dist_b_pct = bench[TARGET_BENCH].value_counts(normalize=True) * 100
ratio_b    = dist_b.max() / dist_b.min()

print("=" * 60)
print(f"BENCHMARK — DISTRIBUTION DE LA CIBLE : {TARGET_BENCH}")
print("=" * 60)
print(pd.DataFrame({'Count': dist_b, 'Pct_%': dist_b_pct.round(2)}).to_string())
print(f"\n⚠️  Ratio déséquilibre max/min : {ratio_b:.1f}x")

# Figure
fig, ax = plt.subplots(figsize=(10, 5))
dist_b.sort_values().plot(kind='barh', ax=ax, color='#C44E52', edgecolor='white', alpha=0.9)
for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}',
                (p.get_width(), p.get_y() + p.get_height() / 2),
                xytext=(4, 0), textcoords='offset points', va='center', fontsize=9)
ax.set_title('Benchmark public — Distribution des catégories (queue)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Nombre de tickets')
plt.tight_layout()
plt.savefig('../notebooks/figures/bench_01_distribution_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ figure : notebooks/figures/bench_01_distribution_classes.png')

In [ ]:
# ── Benchmark : longueur des textes & autres axes (étape 5) ──────────────────
bench['text_combined'] = (bench['subject'].fillna('') + ' ' + bench['body'].fillna('')).str.strip()
bench['word_count']    = bench['text_combined'].str.split().apply(len)
bench['char_count']    = bench['text_combined'].str.len()

print("=" * 60)
print("BENCHMARK — STATISTIQUES DE LONGUEUR (subject + body)")
print("=" * 60)
print(bench[['word_count', 'char_count']].describe().round(1).to_string())

print("\n--- Type de ticket (style ITIL) ---")
print(bench['type'].value_counts().to_string())
print("\n--- Priorité ---")
print(bench['priority'].value_counts().to_string())

In [ ]:
# ── Comparaison benchmark vs notre dataset (étape 5) ─────────────────────────
# Recharger notre dataset brut pour une comparaison équitable (texte brut Titre+Description)
own = pd.read_csv(f"../{params['data']['raw_path']}", encoding='utf-8-sig')
own['text_combined'] = (own['Titre'].fillna('') + ' ' + own['Description'].fillna('')).str.strip()
own['word_count']    = own['text_combined'].str.split().apply(len)
dist_o = own['Catégorie'].value_counts()

comparaison = pd.DataFrame({
    'Notre dataset (HelpDesk1)': [
        len(own), own['Catégorie'].nunique(),
        round(dist_o.max() / dist_o.min(), 1), int(own['word_count'].median()),
    ],
    'Benchmark public (EN)': [
        len(bench), bench['queue'].nunique(),
        round(dist_b.max() / dist_b.min(), 1), int(bench['word_count'].median()),
    ],
}, index=['Nb tickets', 'Nb classes', 'Déséquilibre max/min', 'Mots/ticket (médiane)'])

print("=" * 60)
print("COMPARAISON — NOTRE DATASET vs BENCHMARK PUBLIC")
print("=" * 60)
print(comparaison.to_string())

# Figure comparative
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(own['word_count'],   bins=30, color='#4C72B0', alpha=0.85, edgecolor='white',
             density=True, label='Notre dataset')
axes[0].hist(bench['word_count'], bins=30, color='#C44E52', alpha=0.55, edgecolor='white',
             density=True, label='Benchmark')
axes[0].set_xlim(0, 150)
axes[0].set_title('Longueur des tickets (mots) — densité', fontweight='bold')
axes[0].set_xlabel('Nombre de mots'); axes[0].legend()

x = range(len(comparaison.columns))
axes[1].bar(x, comparaison.loc['Déséquilibre max/min'], color=['#4C72B0', '#C44E52'], alpha=0.85)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(comparaison.columns, fontsize=9)
axes[1].set_title('Déséquilibre des classes (max/min)', fontweight='bold')
for i, v in enumerate(comparaison.loc['Déséquilibre max/min']):
    axes[1].text(i, v, f'{v}x', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/figures/bench_02_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ figure : notebooks/figures/bench_02_comparaison.png')

print("\n" + "=" * 60)
print("LECTURE")
print("=" * 60)
print(f"- Notre corpus est plus compact (médiane {int(own['word_count'].median())} mots) "
      f"et mieux équilibré ({dist_o.max()/dist_o.min():.1f}x) que le benchmark "
      f"({dist_b.max()/dist_b.min():.1f}x).")
print("- Le benchmark public confirme la pertinence de la tâche : classification "
      "multi-classes de tickets sur texte court (Titre + Description).")
print("- Pistes : évaluer nos modèles sur ce benchmark (transfert), ou enrichir le vocabulaire IT.")